# 01 · Diagnóstico da sessão GPU para o piloto de 200 patentes

**Finalidade.** Verificar se a sessão pode executar os componentes do piloto Qwen3-8B + LoRA e multilingual-e5-large. Os testes usam entradas sintéticas. O relatório registra o ambiente e o perfil do ajuste em `resultados/etapa_1/piloto_200/diagnostico_gpu_<UTC>.json`.

1. Abra a cópia atualizada de **Meu Drive** e selecione **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU**. A imagem de referência é **2026.01**, com Python 3.12.12 e PyTorch 2.9.0. Em outra imagem, o diagnóstico registra as versões efetivas.
2. Execute as células em ordem. Se a instalação modificar pacotes, escolha **Reiniciar sessão** e **Executar tudo** novamente na mesma máquina virtual.
3. Deixe as opções de testes completos desmarcadas na primeira passagem. Os testes padrão consultam arquivos pequenos dos modelos e exercitam uma arquitetura Qwen3 reduzida. Depois, se quiser medir o custo de um microlote real do 8B, habilite `TESTAR_CAPACIDADE_QWEN`. O download inicial dos pesos BF16 ocupa cerca de **16,4 GB**. O teste independente `TESTAR_E5_COMPLETO` baixa cerca de **2,24 GB** de pesos. Ambos usam o disco temporário do Colab.

O resultado de qualidade da extração e o efeito da adaptação ao CPC serão avaliados nas patentes coletadas nas próximas etapas. [Ambiente Colab](https://research.google.com/colaboratory/runtime-version-faq.html) · [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B/tree/b968826d9c46dd6066d109eabc6255188de91218)

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PASTA_DRIVE = "/content/drive/MyDrive/Replicação LLM EIDT IC"  # @param {type:"string"}
TESTAR_CAPACIDADE_QWEN = False  # @param {type:"boolean"}
TESTAR_E5_COMPLETO = False  # @param {type:"boolean"}
raiz = Path(PASTA_DRIVE).expanduser().resolve()
if not raiz.is_relative_to(Path("/content/drive").resolve()):
    raise ValueError("Escolha o caminho do projeto no Drive montado.")
arquivo_requisitos = raiz / "requirements.txt"
arquivo_config = raiz / "config" / "experimento.json"
for arquivo in (arquivo_requisitos, arquivo_config):
    if not arquivo.is_file():
        raise FileNotFoundError(f"Confira a cópia da pasta completa do projeto. Falta {arquivo}")

### 1 · Verificar a sessão antes da instalação

O teste interrompe a instalação quando não há GPU CUDA. A cada execução, o relatório registra horário, configuração, versões, memória e resultado dos testes. O inventário contém apenas nomes e versões de pacotes.

In [ ]:
from contextlib import contextmanager
from datetime import datetime, timezone
from hashlib import sha256
from importlib import metadata
from pathlib import Path
import gc
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile

os.environ["USE_TF"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

def agora():
    return datetime.now(timezone.utc)

pasta_saida = raiz / "resultados" / "etapa_1" / "piloto_200"
pasta_saida.mkdir(parents=True, exist_ok=True)
arquivo_saida = pasta_saida / f"diagnostico_gpu_{agora():%Y%m%dT%H%M%S%fZ}.json"
config = json.loads(arquivo_config.read_text(encoding="utf-8-sig"))
cache_modelos = Path("/content/eidt_modelos_hf")
relatorio = {
    "inicio_utc": agora().isoformat(),
    "piloto_id": config["piloto"]["id"],
    "estado": "em_andamento",
    "sha256_config": sha256(arquivo_config.read_bytes()).hexdigest(),
    "sha256_requisitos": sha256(arquivo_requisitos.read_bytes()).hexdigest(),
    "testes": {},
}

def salvar():
    relatorio["atualizado_utc"] = agora().isoformat()
    temporario = arquivo_saida.with_suffix(".tmp")
    temporario.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2), encoding="utf-8")
    temporario.replace(arquivo_saida)

@contextmanager
def teste(nome):
    relatorio["testes"][nome] = {"estado": "em_andamento"}
    salvar()
    try:
        yield relatorio["testes"][nome]
    except Exception as erro:
        relatorio["testes"][nome].update(estado="falhou", tipo_erro=type(erro).__name__)
        relatorio["estado"] = "requer_atencao"
        print(f"Confira a exceção nesta célula e o relatório {arquivo_saida.name}.")
        raise
    else:
        relatorio["testes"][nome]["estado"] = "passou"
    finally:
        salvar()

with teste("sessao_cuda") as dados:
    import torch
    from packaging.version import Version

    dados.update(python=platform.python_version(), torch=str(torch.__version__),
                 cuda_pytorch=torch.version.cuda, sistema=platform.platform())
    referencia = sys.version_info[:2] == (3, 12) and Version(torch.__version__).base_version == "2.9.0"
    dados["perfil_python_torch"] = "referencia" if referencia else "alternativo"
    print("Python", dados["python"], "| PyTorch", dados["torch"], "| CUDA", dados["cuda_pytorch"])
    if not (3, 10) <= sys.version_info[:2] <= (3, 13) or Version(torch.__version__) < Version("2.4"):
        raise RuntimeError("Selecione a imagem GPU de referência 2026.01 antes de instalar as bibliotecas.")
    if not torch.cuda.is_available():
        raise RuntimeError("Ative uma GPU no menu Ambiente de execução e execute novamente.")
    memoria_livre, memoria_total = torch.cuda.mem_get_info()
    dados.update(gpu=torch.cuda.get_device_name(0), capacidade_cuda=list(torch.cuda.get_device_capability(0)),
                 bf16_nativo=torch.cuda.is_bf16_supported(including_emulation=False), vram_livre_bytes=memoria_livre,
                 vram_total_gib=round(memoria_total / 2**30, 2), vram_livre_gib=round(memoria_livre / 2**30, 2),
                 disco_livre_gib=round(shutil.disk_usage("/content").free / 2**30, 2))
    if shutil.which("nvidia-smi"):
        dados["driver"] = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"], text=True
        ).strip().splitlines()[0]
    print(dados["gpu"], "| VRAM livre", dados["vram_livre_gib"], "GiB |", dados["perfil_python_torch"])
print("Relatório desta execução:", arquivo_saida)

### 2 · Instalar as versões do projeto

O instalador preserva o PyTorch e o NumPy da imagem. Havendo mudança de pacote, o relatório ficará marcado como `reinicio_necessario`. **Reinicie a sessão e execute o notebook desde o começo**. A execução seguinte grava seu próprio relatório.

In [ ]:
if globals().get("reinicio_pendente", False):
    raise RuntimeError("Reinicie a sessão antes de rodar as células novamente.")

with teste("instalacao") as dados:
    with tempfile.TemporaryDirectory(dir="/content") as pasta:
        restricoes = Path(pasta) / "base_colab.txt"
        plano_pip = Path(pasta) / "operacoes_pip.json"
        restricoes.write_text(
            f"torch=={metadata.version('torch')}\nnumpy=={metadata.version('numpy')}\n",
            encoding="utf-8",
        )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "--progress-bar", "off", "-c", str(restricoes), "-r", str(arquivo_requisitos),
             "--report", str(plano_pip)],
            check=True,
        )
        operacoes = json.loads(plano_pip.read_text(encoding="utf-8"))["install"]
        dados["pacotes_alterados"] = sorted(item["metadata"]["name"] for item in operacoes)
    reinicio_pendente = bool(operacoes)

if reinicio_pendente:
    relatorio["estado"] = "reinicio_necessario"
    salvar()
    raise RuntimeError("Pacotes instalados. Escolha Reiniciar sessão e depois Executar tudo.")

### 3 · Conferir bibliotecas, perfil do piloto e revisões dos modelos

O Hub fornece os metadados, as configurações e o índice de pesos da revisão registrada em `config/experimento.json`. Os tamanhos são lidos do índice do próprio 8B. O teste compara os módulos LoRA com os nomes das matrizes de pesos e registra a memória disponível para os testes opcionais.

In [ ]:
if globals().get("reinicio_pendente", False) or relatorio["testes"].get("instalacao", {}).get("estado") != "passou":
    raise RuntimeError("Conclua a instalação e o reinício indicado antes desta célula.")

with teste("bibliotecas") as dados:
    import accelerate
    import bitsandbytes as bnb
    import peft
    import sentencepiece
    import transformers
    from huggingface_hub import HfApi, hf_hub_download
    from sklearn.cluster import KMeans
    from umap import UMAP
    from transformers import AutoConfig

    dados["versoes_diretas"] = {}
    for linha in arquivo_requisitos.read_text(encoding="utf-8-sig").splitlines():
        linha = linha.split("#", 1)[0].strip()
        if linha:
            pacote, esperada = linha.split("==")
            atual = metadata.version(pacote)
            if atual != esperada:
                raise RuntimeError(f"{pacote} tem versão {atual}. O protocolo especifica {esperada}.")
            dados["versoes_diretas"][pacote] = atual
    relatorio["inventario"] = dict(sorted(
        (dist.metadata["Name"], dist.version) for dist in metadata.distributions() if dist.metadata.get("Name")
    ))
    print(dados["versoes_diretas"])

with teste("revisoes_hub") as dados:
    hub = HfApi(token=False)
    for chave, identidade, tipo in [
        ("qwen", "Qwen/Qwen3-8B", "qwen3"),
        ("embeddings", "intfloat/multilingual-e5-large", "xlm-roberta"),
    ]:
        modelo = config["modelos"][chave]
        repo, revisao = modelo["id"], modelo["revisao"]
        if repo != identidade or not re.fullmatch(r"[0-9a-f]{40}", revisao):
            raise ValueError(f"Confira o modelo e a revisão completa em modelos.{chave}.")
        info = hub.model_info(repo, revision=revisao, timeout=30)
        if info.sha != revisao:
            raise RuntimeError(f"O Hub devolveu outra revisão para {repo}.")
        arquitetura = AutoConfig.from_pretrained(repo, revision=revisao, token=False, trust_remote_code=False,
                                                 cache_dir=cache_modelos)
        if arquitetura.model_type != tipo:
            raise ValueError(f"Arquitetura inesperada para {repo}.")
        dados[chave] = dict(id=repo, revisao=info.sha, arquitetura=arquitetura.model_type,
                            dimensao_oculta=arquitetura.hidden_size)
    qwen = config["modelos"]["qwen"]
    indice = hf_hub_download(qwen["id"], "model.safetensors.index.json", revision=qwen["revisao"],
                             token=False, cache_dir=cache_modelos)
    mapa = json.loads(Path(indice).read_text())
    bytes_pesos = mapa["metadata"]["total_size"]
    dados["qwen"]["pesos_originais_bytes"] = bytes_pesos
    dados["qwen"]["pesos_originais_gib"] = round(bytes_pesos / 2**30, 2)
    print("Pesos originais do Qwen:", dados["qwen"]["pesos_originais_gib"], "GiB")
    print("A VRAM adicional depende das ativações, dos adaptadores e dos estados do otimizador.")

with teste("parametros_piloto") as dados:
    treino = config["piloto"]["treino"]
    campos = ("r", "alpha", "dropout", "target_modules", "max_length", "microbatch", "acumulacao",
              "batch_efetivo", "precisao", "quantizacao", "gradient_checkpointing", "num_gpus")
    dados.update({nome: treino[nome] for nome in campos})
    if treino["precisao"] != "bf16" or treino["quantizacao"] is not None or treino["num_gpus"] != 1:
        raise ValueError("O teste de capacidade atende o perfil LoRA BF16 em uma GPU do piloto.")
    for chave in ("r", "alpha", "microbatch", "acumulacao", "batch_efetivo", "max_length"):
        if not isinstance(treino[chave], int) or treino[chave] <= 0:
            raise ValueError(f"Valor inválido em piloto.treino.{chave}.")
    if treino["microbatch"] * treino["acumulacao"] != treino["batch_efetivo"]:
        raise ValueError("Em uma GPU, o lote efetivo deve ser o microlote vezes a acumulação.")
    if not 0 <= treino["dropout"] < 1:
        raise ValueError("Revise o dropout dos adaptadores.")
    modulos = treino["target_modules"]
    pesos_camada_0 = [nome for nome in mapa["weight_map"] if nome.startswith("model.layers.0.")]
    if not modulos or any(not any(nome.endswith(f".{modulo}.weight") for nome in pesos_camada_0) for modulo in modulos):
        raise ValueError("Há módulo LoRA ausente no índice da revisão do Qwen3-8B.")
    print("Perfil de treino em uma GPU:", dados)

with teste("perfil_extracao") as dados:
    from transformers import AutoTokenizer, GenerationConfig

    extracao = config["piloto"]["extracao"]
    if extracao["modelo"] != qwen["id"] or extracao["do_sample"] or extracao["enable_thinking"]:
        raise ValueError("Confira o perfil de extração determinística do Qwen3-8B.")
    tokenizador_qwen = AutoTokenizer.from_pretrained(qwen["id"], revision=qwen["revisao"],
                                                     token=False, trust_remote_code=False, cache_dir=cache_modelos)
    exemplo = [{"role": "user", "content": "提取以下示例中的技术要素。标题：细胞检测。摘要：利用荧光检测细胞。"}]
    ids_teste = tokenizador_qwen.apply_chat_template(exemplo, tokenize=True, add_generation_prompt=True,
                                                     enable_thinking=extracao["enable_thinking"])
    geracao = GenerationConfig(do_sample=extracao["do_sample"], max_new_tokens=extracao["max_new_tokens"])
    janela_modelo = AutoConfig.from_pretrained(qwen["id"], revision=qwen["revisao"],
                                              token=False, trust_remote_code=False, cache_dir=cache_modelos).max_position_embeddings
    if (extracao["lote"] != 1 or not 0 < len(ids_teste) <= extracao["max_entrada"]
            or extracao["max_entrada"] + geracao.max_new_tokens > janela_modelo):
        raise ValueError("O orçamento de tokens da extração é incompatível com a configuração do modelo.")
    dados.update(modelo=qwen["id"], revisao=qwen["revisao"], entrada_sintetica_tokens=len(ids_teste),
                 lote=extracao["lote"], max_entrada=extracao["max_entrada"], max_new_tokens=geracao.max_new_tokens,
                 do_sample=geracao.do_sample, enable_thinking=extracao["enable_thinking"])
    print("Perfil de extração e tokenização do exemplo chinês conferidos.")

### 4 · Exercitar os adaptadores e o kernel NF4

O Qwen3 reduzido usa os módulos, o posto, o escalonamento e o dropout que o piloto define no JSON. A passagem com oito tokens verifica gradientes nos adaptadores a baixo custo. Uma camada NF4 separada exercita a biblioteca bitsandbytes para uma eventual comparação futura. O perfil principal do piloto usa BF16. Em uma GPU sem BF16 nativo, o teste de integração reduzido usa FP16 e o relatório registra a limitação do perfil principal.

In [ ]:
with teste("lora_gpu_sintetico") as dados:
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import Qwen3Config, Qwen3ForCausalLM

    torch.manual_seed(42)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
    mini_config = Qwen3Config(
        vocab_size=128, hidden_size=64, intermediate_size=128, head_dim=16,
        num_hidden_layers=1, num_attention_heads=4, num_key_value_heads=2,
        max_position_embeddings=64, pad_token_id=0, bos_token_id=1, eos_token_id=2,
    )
    mini_qwen = Qwen3ForCausalLM(mini_config).to(device="cuda", dtype=dtype)
    adaptador = LoraConfig(task_type=TaskType.CAUSAL_LM, r=treino["r"], lora_alpha=treino["alpha"],
                           lora_dropout=treino["dropout"], target_modules=treino["target_modules"])
    mini_qwen = get_peft_model(mini_qwen, adaptador)
    treinaveis = [nome for nome, p in mini_qwen.named_parameters() if p.requires_grad]
    if not treinaveis or any("lora_" not in nome for nome in treinaveis):
        raise RuntimeError("A separação entre adaptadores e pesos congelados falhou.")
    mini_qwen.train()
    tokens = torch.arange(8, device="cuda")[None, :]
    perda = mini_qwen(tokens, labels=tokens, use_cache=False).loss
    if not torch.isfinite(perda).item():
        raise RuntimeError("O Qwen sintético produziu uma perda não finita.")
    perda.backward()
    if any(p.grad is None for p in mini_qwen.parameters() if p.requires_grad):
        raise RuntimeError("Um adaptador do Qwen sintético ficou sem gradiente.")
    dados.update(dtype=str(dtype), tensores_lora_treinaveis=len(treinaveis), perda_sintetica=perda.item(),
                 atualizado_pelo_otimizador=False)
    del tokens, perda, mini_qwen
    print("LoRA exercitado com oito tokens sintéticos e gradientes nos adaptadores.")

with teste("kernel_nf4") as dados:
    camada = bnb.nn.Linear4bit(64, 64, bias=False, compute_dtype=dtype, quant_type="nf4").to("cuda")
    with torch.inference_mode():
        saida = camada(torch.randn(2, 64, device="cuda", dtype=dtype))
        if camada.weight.quant_state is None or saida.shape != (2, 64) or not torch.isfinite(saida).all().item():
            raise RuntimeError("A camada quantizada produziu uma saída inválida.")
    dados.update(tipo_quantizacao="nf4", formato=list(saida.shape))
    del camada, saida
    print("Kernel NF4 executado na GPU.")

### 5 · Estimar o perfil BF16 e medir um microlote opcional

O relatório compara a VRAM livre com os pesos do índice oficial. Para exercitar o **8B real** nesta GPU, marque `TESTAR_CAPACIDADE_QWEN` e execute as células em ordem. Reserve espaço para ~16,4 GB de pesos no cache local. A célula também verifica a RAM disponível, o BF16 nativo e alguma folga de VRAM. Essas verificações são condições de partida. Ao carregar o modelo, a cópia é direcionada à GPU e os adaptadores permanecem treináveis.

O teste executa um único microlote sintético no comprimento configurado para o ajuste CPC, calcula o gradiente e registra o pico de memória. Os pesos são descartados da GPU ao final. Estados de otimizador, acumulação completa, validação, extração com geração e qualidade do resultado exigem medições posteriores.

In [ ]:
gpu = relatorio["testes"]["sessao_cuda"]
pesos = relatorio["testes"]["revisoes_hub"]["qwen"]
limitacoes = []
if not gpu["bf16_nativo"]:
    limitacoes.append("bf16_nativo_indisponivel")
if gpu["vram_livre_bytes"] < pesos["pesos_originais_bytes"]:
    limitacoes.append("vram_livre_abaixo_dos_pesos")
perfil = {
    "estado": "perfil_bf16_inviavel_nesta_sessao" if limitacoes else "piso_de_pesos_atendido",
    "limitacoes": limitacoes,
    "pesos_originais_gib": pesos["pesos_originais_gib"],
    "vram_livre_inicial_gib": gpu["vram_livre_gib"],
    "microbatch": treino["microbatch"],
    "max_length": treino["max_length"],
    "treino_completo": "a_validar_no_piloto",
}
relatorio["perfil_lora_8b"] = perfil
salvar()
print("Perfil do 8B:", perfil)

if TESTAR_CAPACIDADE_QWEN:
    from huggingface_hub import try_to_load_from_cache
    from psutil import virtual_memory
    from transformers import AutoModelForCausalLM

    with teste("microlote_qwen8b_real") as dados:
        if limitacoes:
            raise RuntimeError(f"O teste BF16 precisa de outra GPU. Limitações: {limitacoes}")
        shards = set(mapa["weight_map"].values())
        ja_em_cache = all(isinstance(try_to_load_from_cache(qwen["id"], nome, cache_dir=cache_modelos,
                                                             revision=qwen["revisao"]), str) for nome in shards)
        tamanho = pesos["pesos_originais_bytes"]
        disco_exigido = 2 * 2**30 + (0 if ja_em_cache else tamanho)
        ram_livre = virtual_memory().available
        dados.update(ram_livre_gib=round(ram_livre / 2**30, 2), cache_completo=ja_em_cache)
        if shutil.disk_usage("/content").free < disco_exigido or ram_livre < 5 * 2**30:
            raise RuntimeError("Reserve o espaço dos pesos mais 2 GiB de disco e pelo menos 5 GiB de RAM livre.")
        if torch.cuda.mem_get_info()[0] < tamanho + 2 * 2**30:
            raise RuntimeError("O teste requer os pesos mais uma folga mínima de 2 GiB de VRAM livre.")
        if not 2 <= treino["max_length"] <= 40960:
            raise ValueError("Comprimento de teste incompatível com o Qwen3-8B.")

        def medir_microlote():
            base = modelo = tokens = perda = None
            try:
                base = AutoModelForCausalLM.from_pretrained(
                    qwen["id"], revision=qwen["revisao"], token=False, trust_remote_code=False,
                    use_safetensors=True, dtype=torch.bfloat16, low_cpu_mem_usage=True,
                    device_map={"": 0}, attn_implementation="sdpa", cache_dir=cache_modelos,
                )
                base.config.use_cache = False
                modelo = get_peft_model(base, adaptador).train()
                if treino["gradient_checkpointing"]:
                    modelo.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
                tokens = torch.randint(0, base.config.vocab_size, (treino["microbatch"], treino["max_length"]),
                                       device="cuda")
                perda = modelo(input_ids=tokens, labels=tokens, use_cache=False).loss
                if not torch.isfinite(perda).item():
                    raise RuntimeError("A perda do microlote sintético precisa ser finita.")
                perda.backward()
                torch.cuda.synchronize()
                if any(p.grad is None for p in modelo.parameters() if p.requires_grad):
                    raise RuntimeError("Há parâmetros treináveis sem gradiente após a retropropagação.")
                return perda.item()
            finally:
                del base, modelo, tokens, perda

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        dados.update(dtype="bf16", microbatch=treino["microbatch"], max_length=treino["max_length"],
                     gradient_checkpointing=treino["gradient_checkpointing"], passos_otimizador=0)
        try:
            dados["perda_sintetica"] = medir_microlote()
            perfil["estado"] = "microlote_bf16_medido"
        finally:
            dados["pico_alocado_gib"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
            dados["pico_reservado_gib"] = round(torch.cuda.max_memory_reserved() / 2**30, 2)
            gc.collect()
            torch.cuda.empty_cache()
        print("Microlote verificado. Pico alocado em GiB:", dados["pico_alocado_gib"])
else:
    relatorio["testes"]["microlote_qwen8b_real"] = {"estado": "opcional_nao_executado"}
    salvar()

### 6 · Teste opcional com embeddings E5 reais

Para habilitá-lo, marque `TESTAR_E5_COMPLETO` na primeira célula. Reserve pelo menos **3 GiB livres no disco local e na GPU** para este teste. O download fica no cache temporário da sessão. Os dois textos chineses são sintéticos e tratam de técnicas de laboratório. Usamos o prefixo `query: ` para a tarefa simétrica de agrupamento, média mascarada dos tokens e normalização L2, conforme a ficha do modelo. [Uso do multilingual-e5-large](https://huggingface.co/intfloat/multilingual-e5-large)

In [ ]:
if TESTAR_E5_COMPLETO:
    with teste("embeddings_e5_reais") as dados:
        from transformers import AutoModel, AutoTokenizer
        import torch.nn.functional as F

        if min(torch.cuda.mem_get_info()[0], shutil.disk_usage("/content").free) < 3 * 2**30:
            raise RuntimeError("Libere pelo menos 3 GiB de armazenamento local e VRAM ou desmarque o teste opcional.")
        e5 = config["modelos"]["embeddings"]
        opcoes = dict(revision=e5["revisao"], trust_remote_code=False, token=False, cache_dir=cache_modelos)
        tokenizador = AutoTokenizer.from_pretrained(e5["id"], **opcoes)
        encoder = AutoModel.from_pretrained(e5["id"], use_safetensors=True, **opcoes).eval().to("cuda")
        entradas = tokenizador(["query: 荧光显微镜检测细胞", "query: 使用酶反应测定蛋白质浓度"],
                               padding=True, truncation=True, max_length=512, return_tensors="pt").to("cuda")
        with torch.inference_mode():
            ocultos = encoder(**entradas).last_hidden_state
            mascara = entradas.attention_mask.unsqueeze(-1).bool()
            medias = ocultos.masked_fill(~mascara, 0).sum(dim=1) / mascara.sum(dim=1)
            vetores = F.normalize(medias.float(), p=2, dim=1)
            normas = vetores.norm(dim=1)
            if vetores.shape != (2, 1024) or not torch.allclose(normas, torch.ones_like(normas), atol=1e-5):
                raise RuntimeError("Forma ou norma incompatível com os embeddings E5 esperados.")
        dados.update(formato=list(vetores.shape), normas_l2=normas.tolist())
        del encoder, ocultos, vetores
        print("Dois vetores de 1.024 dimensões com norma unitária.")
else:
    relatorio["testes"]["embeddings_e5_reais"] = {"estado": "opcional_nao_executado"}
    salvar()
    print("Pesos E5 opcionais dispensados nesta execução.")

### 7 · Ler o resultado

`diagnostico_aprovado` significa que as operações obrigatórias funcionaram. Leia também `perfil_lora_8b` para saber se a sessão atende ao piso de pesos e se o microlote em BF16 foi medido. O campo `embeddings_e5_reais` identifica se houve teste do E5 completo. Guarde o JSON desta execução com o tipo de GPU no projeto local.

In [ ]:
obrigatorios = ["sessao_cuda", "instalacao", "bibliotecas", "revisoes_hub", "parametros_piloto", "perfil_extracao",
                "lora_gpu_sintetico", "kernel_nf4"]
if TESTAR_CAPACIDADE_QWEN:
    obrigatorios.append("microlote_qwen8b_real")
if TESTAR_E5_COMPLETO:
    obrigatorios.append("embeddings_e5_reais")
pendentes = [nome for nome in obrigatorios if relatorio["testes"].get(nome, {}).get("estado") != "passou"]
if pendentes or "perfil_lora_8b" not in relatorio:
    relatorio["estado"] = "requer_atencao"
    salvar()
    raise RuntimeError(f"Execute as seções pendentes em ordem. Testes pendentes: {pendentes}")
relatorio["estado"] = "diagnostico_aprovado"
salvar()
print(relatorio["estado"], "| Python/PyTorch", relatorio["testes"]["sessao_cuda"]["perfil_python_torch"])
print("LoRA BF16 do 8B:", relatorio["perfil_lora_8b"])
print("Teste E5 real:", relatorio["testes"].get("embeddings_e5_reais", {}).get("estado", "opcional_nao_executado"))
print("Relatório no Drive:", arquivo_saida)